<a href="https://colab.research.google.com/github/Damian200211/PharmaRoute/blob/main/PharmaRoute.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install system dependencies: poppler (for pdf2image) and tesseract (OCR)
!apt-get update -qq && apt-get install -y -qq poppler-utils tesseract-ocr

# rest of pip installs...

# Uninstall any conflicting llama-cpp-python, then install GPU binary
!pip uninstall -y llama-cpp-python
!pip install --no-cache-dir --only-binary llama-cpp-python llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

# Python libraries
!pip install -q PyPDF2 langchain-text-splitters llama-index llama-index-embeddings-huggingface gradio pdf2image pytesseract Pillow
!pip install llama-index-llms-llama-cpp
!pip install "uvicorn<0.30.0"
!pip install -q opencv-python-headless

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.9_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.9) ...
Setting up poppler-utils (24.02.0-1ubuntu9.9) ...
Processing triggers for man-db (2.12.0-4build2) ...
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 190.9 MB/s eta 

In [2]:
import os, requests

model_path = "/content/mistral-7b-instruct-v0.2.Q4_K_M.gguf"
if not os.path.exists(model_path):
    print("Downloading Mistral 7B Q4_K_M (~4.1 GB) – please wait...")
    url = "https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf"
    response = requests.get(url, stream=True)
    with open(model_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Download complete.")
else:
    print("Model already downloaded.")

Download complete.


In [3]:
import os, traceback
from typing import List, Tuple
import numpy as np
import gradio as gr

from PyPDF2 import PdfReader
from pdf2image import convert_from_path
import pytesseract
import cv2

from langchain_text_splitters import RecursiveCharacterTextSplitter
from llama_index.core import Document, VectorStoreIndex, Settings, PromptTemplate
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator
from llama_index.core.response_synthesizers import get_response_synthesizer, ResponseMode
from llama_index.core.schema import NodeWithScore

In [4]:
# ==========================================
# 1. SETUP LLM & EMBEDDINGS
# ==========================================
Settings.llm = LlamaCPP(
    model_path=model_path,          # use the pre‑downloaded file
    model_url=None,
    temperature=0.1,
    max_new_tokens=512,
    context_window=4096,
    model_kwargs={"n_gpu_layers": -1},
    generate_kwargs={"stop": ["Question:", "Q:"]},
    verbose=False,
)

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# ==========================================
# 2. CONSTANTS & PROMPTS
# ==========================================
VALID_DOC_TYPES = [
    "Cover Letter", "Certificate Of Quality", "Packaging Specification",
    "Bse/Tse Declaration", "Material Description", "Supplier Qualification",
    "Chain Of Custody", "Safety Data Sheet", "Other"
]

CITATION_PROMPT_TEMPLATE = (
    "You are a precise assistant that answers ONLY the question asked, using the provided context chunks.\n"
    "If the context cannot answer the question, say 'I don't know'.\n"
    "You MUST answer ONLY the single question below. Do NOT generate any follow‑up questions or additional dialogue.\n"
    "Always cite the source using [Chunk X, Pages Y-Z] format.\n"
    "Here are the context chunks:\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Question: {query_str}\n"
    "Answer (single answer, no extra questions): "
)

QA_PROMPT = PromptTemplate(CITATION_PROMPT_TEMPLATE)

In [6]:
# ==========================================
# 3. UTILITY FUNCTIONS
# ==========================================
def safe_llm_predict(prompt: str) -> str:
    safe = prompt.replace("[/INST]", "").replace("[INST]", "")
    formatted = f"[INST] {safe} [/INST]"          # no leading <s>
    try:
        resp = Settings.llm.complete(formatted)
        return resp.text.strip()
    except Exception as e:
        print(f"LLM error: {e}")
        return "Other"

def clean_doc_type(raw: str) -> str:
    cleaned = raw.strip().replace('"','').replace('`','').replace('*','').strip('. ')
    for label in VALID_DOC_TYPES:
        if label.lower() in cleaned.lower():
            return label
    return "Other"

def classify_logical_document(text_snippet: str) -> str:
    prompt = f"""
    You are a pharmaceutical document classifier. Based on the content
    below, classify the document into ONE of these types:

    - Cover Letter: General product information, operating temperatures, and storage conditions.
    - Certificate Of Quality: Lot numbers, manufacture/expiration dates, and test results.
    - Packaging Specification: Packaging components, materials, and configuration changes.
    - Bse/Tse Declaration: Animal-origin declarations and compliance.
    - Material Description: Materials of construction, sterilization compatibility, and physical properties.
    - Supplier Qualification: Supplier audits, ISO certifications, and supply chain locations.
    - Chain Of Custody: Traceability, manufactured assemblies, and shipment flow.
    - Safety Data Sheet: Handling precautions, bulk density, thermal decomposition, and safety hazards.
    - Other: General or unclear queries.

    Content: {text_snippet[:1500]}

    Respond with ONLY the exact document type name. No explanation.
    """
    return clean_doc_type(safe_llm_predict(prompt))

In [7]:
# ==========================================
# 4. PDF EXTRACTION (DIGITAL + OCR)
# ==========================================
def preprocess_image_for_ocr(pil_image):
    """Enhance scanned image for better OCR accuracy."""
    gray = np.array(pil_image.convert("L"))
    bw = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    return bw

def extract_text_from_page(page, pdf_path, page_idx: int) -> Tuple[str, bool]:
    """Extract text; returns (text, is_scanned). If PyPDF2 gives text, use it; else OCR."""
    text = page.extract_text()
    if text and text.strip():
        return text.strip(), False
    # Try OCR with preprocessing
    try:
        images = convert_from_path(pdf_path, first_page=page_idx+1, last_page=page_idx+1, dpi=200)
        if images:
            processed = preprocess_image_for_ocr(images[0])
            ocr_text = pytesseract.image_to_string(processed)
            return ocr_text.strip(), True
    except Exception as e:
        print(f"OCR failed for page {page_idx}: {e}")
    return "", False

In [8]:
# ==========================================
# 5. DOCUMENT SEGMENTATION
# ==========================================
def compute_embedding(text: str) -> np.ndarray:
    return np.array(Settings.embed_model.get_text_embedding(text))

def segment_pages_into_documents(pages: List[dict], threshold=0.75) -> List[List[dict]]:
    if not pages:
        return []
    embeddings = [compute_embedding(p["text"][:200]) for p in pages]
    docs = []
    cur = [pages[0]]
    prev_emb = embeddings[0]
    for i in range(1, len(pages)):
        sim = np.dot(embeddings[i], prev_emb) / (np.linalg.norm(embeddings[i]) * np.linalg.norm(prev_emb) + 1e-9)
        if sim < threshold:
            docs.append(cur)
            cur = [pages[i]]
        else:
            cur.append(pages[i])
        prev_emb = embeddings[i]
    docs.append(cur)
    return docs

In [9]:
# ==========================================
# 6. SYNCHRONOUS INDEXING
# ==========================================
def build_index_from_pdf(pdf_file):
    """Process PDF and build index synchronously. Returns (index, status_msg)."""
    if pdf_file is None:
        return None, " Please upload a PDF first."

    file_path = pdf_file.name
    file_name = os.path.basename(file_path)

    try:
        reader = PdfReader(file_path)
    except Exception as e:
        return None, f" Cannot read PDF: {str(e)}"

    # Extract all pages
    pages = []
    for i, page in enumerate(reader.pages):
        try:
            text, is_scanned = extract_text_from_page(page, file_path, i)
        except Exception as e:
            print(f"Page {i} extraction error: {e}")
            continue
        if text:
            pages.append({
                "page_num": i,
                "text": text,
                "scanned": is_scanned
            })
    if not pages:
        return None, " No readable text found (digital or OCR)."

    # Segment into logical documents
    doc_groups = segment_pages_into_documents(pages, threshold=0.75)

    # Classify each logical document (synchronous, one at a time)
    doc_types = []
    for group in doc_groups:
        combined = group[0]["text"][:800]
        if len(group) > 1:
            combined += "\n...\n" + group[-1]["text"][:200]
        dtype = classify_logical_document(combined)
        doc_types.append(dtype)

    # Build logical doc objects
    logical_docs = []
    for group, dtype in zip(doc_groups, doc_types):
        full_text = "\n\n".join(p["text"] for p in group)
        logical_docs.append({
            "text": full_text,
            "doc_type": dtype,
            "page_start": group[0]["page_num"],
            "page_end": group[-1]["page_num"],
            "scanned": any(p["scanned"] for p in group),
        })

    # Chunk text
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=512, chunk_overlap=100,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    all_documents = []
    for doc_id, ldoc in enumerate(logical_docs):
        chunks = splitter.split_text(ldoc["text"])
        for chunk_idx, chunk in enumerate(chunks):
            all_documents.append(
                Document(
                    text=chunk,
                    metadata={
                        "doc_type": ldoc["doc_type"],
                        "chunk_index": chunk_idx,
                        "doc_id": f"doc_{doc_id}",
                        "page_start": ldoc["page_start"],
                        "page_end": ldoc["page_end"],
                        "source_file": file_name,
                        "scanned": ldoc["scanned"],
                    }
                )
            )

    # Build index
    try:
        index = VectorStoreIndex.from_documents(all_documents)
    except Exception as e:
        return None, f" Indexing failed: {str(e)}"

    status = (
        f" Indexed '{file_name}'\n"
        f"• Pages processed: {len(pages)}\n"
        f"• Logical documents: {len(logical_docs)} (types: {', '.join(set(doc_types))})\n"
        f"• Chunks created: {len(all_documents)}\n"
        f"• OCR pages: {sum(1 for p in pages if p['scanned'])}"
    )
    return index, status

In [10]:
# ==========================================
# 7. QUERY LOGIC
# ==========================================
def predict_doc_type_for_query(query: str) -> str:
    prompt = f"""
    Which pharmaceutical document type is most likely to answer this query: "{query}"?

    Use these definitions to choose the BEST category:
    - Cover Letter: General product information, operating temperatures, and storage conditions.
    - Certificate Of Quality: Lot numbers, manufacture/expiration dates, and test results.
    - Packaging Specification: Packaging components, materials, and configuration changes.
    - Bse/Tse Declaration: Animal-origin declarations and compliance.
    - Material Description: Materials of construction, sterilization compatibility, and physical properties.
    - Supplier Qualification: Supplier audits, ISO certifications, and supply chain locations.
    - Chain Of Custody: Traceability, manufactured assemblies, and shipment flow.
    - Safety Data Sheet: Handling precautions, bulk density, thermal decomposition, and safety hazards.
    - Other: General or unclear queries.

    Respond with ONLY the exact document type name. No explanation.
    """
    return clean_doc_type(safe_llm_predict(prompt))

def process_query(query: str, index, history_state: list) -> Tuple[str, list]:
    # If no index yet, show warning
    if index is None:
        history_state.append({"role": "user", "content": query})
        history_state.append({"role": "assistant", "content": " Please upload and index a PDF first."})
        return "", history_state

    if not query.strip():
        return "", history_state

    predicted_type = predict_doc_type_for_query(query)
    retriever = index.as_retriever(
        similarity_top_k=4,
        filters=MetadataFilters(filters=[
            MetadataFilter(key="doc_type", value=predicted_type, operator=FilterOperator.EQ)
        ])
    )
    raw_results = retriever.retrieve(query)

    used_fallback = False
    if not raw_results:
        retriever = index.as_retriever(similarity_top_k=4)
        raw_results = retriever.retrieve(query)
        used_fallback = True

    if not raw_results:
        bot_msg = " No relevant content found."
    else:
        # Normalize to NodeWithScore
        if hasattr(raw_results[0], 'node'):
            nodes_for_synthesis = raw_results
        else:
            nodes_for_synthesis = [NodeWithScore(node=n, score=0.0) for n in raw_results]

        # Scores
        scores = [round(r.score*100, 1) if r.score else 0.0 for r in raw_results]
        avg_confidence = round(sum(scores)/len(scores), 1) if scores else 0.0

        synthesizer = get_response_synthesizer(
            response_mode=ResponseMode.COMPACT,
            text_qa_template=QA_PROMPT,
        )
        response = synthesizer.synthesize(query, nodes=nodes_for_synthesis)

        bot_msg = str(response.response)
        if used_fallback:
            bot_msg = f"ℹ No chunks under '{predicted_type}', using full document.\n\n{bot_msg}"
        bot_msg += (
            f"\n\n---\n"
            f"**Retrieved chunks:** {len(nodes_for_synthesis)}  |  "
            f"**Avg. confidence:** {avg_confidence}%\n"
            f"**Category:** {predicted_type}" + (" (fallback)" if used_fallback else "")
        )

    # Append to chat history in new format
    history_state.append({"role": "user", "content": query})
    history_state.append({"role": "assistant", "content": bot_msg})
    return "", history_state

In [11]:
# ==========================================
# 8. GRADIO UI
# ==========================================
with gr.Blocks(title="Full‑Stack RAG Chatbot") as demo:
    gr.Markdown("#  Pharma Document Intelligence Chatbot")
    gr.Markdown("Upload a PDF (digital or scanned). Ask questions. Get answers with citations, confidence scores, and full chat history.")

    vector_index_state = gr.State(None)
    chat_history_state = gr.State([])

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label=" Drag & Drop PDF", file_types=[".pdf"], file_count="single")
            process_btn = gr.Button(" Process & Index PDF", variant="primary")
            status_output = gr.Textbox(label="Indexing Status", lines=5, interactive=False)

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat", height=500)
            with gr.Row():
                msg_box = gr.Textbox(label="Your question", placeholder="e.g., Were there any packaging configuration changes?", scale=4)
                submit_btn = gr.Button("Send", variant="secondary", scale=1)
            clear_btn = gr.Button("Clear Chat History")

    process_btn.click(
        fn=build_index_from_pdf,
        inputs=[pdf_input],
        outputs=[vector_index_state, status_output]
    ).then(
        lambda: [], None, [chat_history_state]
    ).then(
        lambda: [], None, [chatbot]
    )

    submit_btn.click(
        fn=process_query,
        inputs=[msg_box, vector_index_state, chat_history_state],
        outputs=[msg_box, chatbot]
    )

    msg_box.submit(
        fn=process_query,
        inputs=[msg_box, vector_index_state, chat_history_state],
        outputs=[msg_box, chatbot]
    )

    clear_btn.click(
        fn=lambda: [], inputs=[], outputs=[chatbot]
    ).then(
        lambda: [], None, [chat_history_state]
    )

print(" Launching Gradio App...")
demo.launch(share=True, debug=True, theme=gr.themes.Soft())

 Launching Gradio App...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://55c4b1625eef930eac.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://55c4b1625eef930eac.gradio.live
